# Surrogate Factory — UCFatigue
## Chapter 7. Model Training
Objectives:
- Train one MLPRegressor per fatigue output.
- Save trained models to artifacts folder.

### 0. Workflow initialisation

In [ ]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

### 7. Model Training

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
workflow.import_metadata(stage_name="SF_7_Model_Training")

In [ ]:
Train_set = workflow.load_data(workflow.config['job_name'] + '_Train_set.csv')
Val_set   = workflow.load_data(workflow.config['job_name'] + '_Val_set.csv')

In [ ]:
from model_training.learn import train
models = train(workflow, Train_set, Val_set)

#### Training Curves
Loss curve per epoch (MLP) and train score per boosting iteration (GradientBoosting).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

models_info = workflow.metadata.get_step_data(['metadata', 'Model_Training', 'Models'])

for info in models_info:
    label = info['label']
    outputs = info['outputs']
    model = joblib.load(info['file'])

    if hasattr(model, 'loss_curve_'):  # MLP
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(model.loss_curve_, label='Training loss', color='steelblue')
        if model.best_loss_ is not None:
            ax.axhline(model.best_loss_, color='r', linestyle='--',
                       label=f'Best loss: {model.best_loss_:.5f}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE Loss')
        ax.set_title(f'{label} — Loss Curve  (stopped at epoch {model.n_iter_} / {model.max_iter})')
        ax.legend()
        plt.tight_layout()
        plt.show()
        plt.close(fig)

    elif hasattr(model, 'estimators_'):  # MultiOutputRegressor (GB)
        ncols = min(4, len(outputs))
        nrows = -(-len(outputs) // ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3*nrows), sharey=False)
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
        n_estimators = model.estimators_[0].n_estimators
        for i, (est, col) in enumerate(zip(model.estimators_, outputs)):
            axes[i].plot(est.train_score_, color='tomato', linewidth=1)
            axes[i].set_title(col, fontsize=9)
            axes[i].set_xlabel('Iteration', fontsize=8)
            axes[i].set_ylabel('MSE Loss', fontsize=8)
        for j in range(len(outputs), len(axes)):
            axes[j].set_visible(False)
        fig.suptitle(f'{label} — Train Score per Boosting Iteration  (n_estimators={n_estimators})', fontsize=12)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

### Save

In [ ]:
workflow.save_metadata()